
# R-ViHSD — ViSoBERT + TF-IDF/LinearSVM Stacking

Notebook này triển khai pipeline:

```text
TEXT
 ├─ ViSoBERT -> P(CLEAN, OFFENSIVE, HATE)
 ├─ char+word TF-IDF + LinearSVM -> 3 decision scores
 ├─ char+word TF-IDF + LinearSVM noise -> 7 noise scores
 └─ handcrafted surface features
          ↓
    Logistic Regression meta-model
          ↓
 CLEAN / OFFENSIVE / HATE
```

`noise_type` được dự đoán bằng TF-IDF + LinearSVM riêng.

### Điểm chính
- OOF prediction để train meta-model, tránh leakage.
- `StratifiedGroupKFold` với group text chuẩn hóa để giảm duplicate/augmentation leakage.
- ViSoBERT dùng weighted cross-entropy + label smoothing để xử lý class imbalance.
- TF-IDF dùng cả word n-gram và character n-gram để chịu teencode/obfuscation/char-repeat.
- Có cache để chạy lại không phải train từ đầu.
- Cuối notebook tạo:
  - `task1_public_output.csv`
  - `task1_private_output.csv`
  - file `.zip` tương ứng.



In [ ]:

# Cài thư viện (Colab/Kaggle). Bỏ comment nếu môi trường chưa có.
!pip -q install -U "transformers>=4.46" "accelerate>=1.0" "scikit-learn>=1.4" sentencepiece joblib


## 1. Cấu hình

In [ ]:

from pathlib import Path

# ===== PATH =====
# Có thể điền đường dẫn trực tiếp. Nếu để None, notebook sẽ tự tìm theo tên file.
TRAIN_CSV = None
VAL_CSV = None
PUBLIC_CSV = None
PRIVATE_CSV = None

WORK_DIR = Path("./rvihsd_stacking_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ===== DATA =====
USE_VALIDATION_FOR_FINAL_TRAIN = True   # final model học train + validation
N_FOLDS = 5                              # giảm xuống 3 nếu GPU/time hạn chế
SEED = 42

# ===== ViSoBERT =====
MODEL_NAME = "uitnlp/visobert"
MAX_LENGTH = 128
EPOCHS = 3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
GRAD_ACCUM_STEPS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.08
LABEL_SMOOTHING = 0.05
USE_CLASS_WEIGHTS = True

# ===== TF-IDF + LinearSVM =====
CHAR_NGRAM = (3, 5)
WORD_NGRAM = (1, 2)
CHAR_MAX_FEATURES = 150_000
WORD_MAX_FEATURES = 80_000
MIN_DF = 2
SVM_C_HATE = 2.0
SVM_C_NOISE = 2.0

# ===== META MODEL =====
META_C = 1.0

# ===== CACHE =====
REUSE_CACHE = True

print("WORK_DIR:", WORK_DIR.resolve())


## 2. Import + seed + tìm file dữ liệu

In [ ]:

import os, re, gc, json, math, random, hashlib, unicodedata, inspect, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

from scipy.special import softmax
from scipy import sparse

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import joblib

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

def auto_find_file(explicit_path, names):
    if explicit_path is not None:
        p = Path(explicit_path)
        if not p.exists():
            raise FileNotFoundError(p)
        return p

    roots = [
        Path("."),
        Path("./data"),
        Path("/content"),
        Path("/content/data"),
        Path("/mnt/data"),
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]
    # Ưu tiên match trực tiếp
    for root in roots:
        for name in names:
            p = root / name
            if p.exists():
                return p

    # Sau đó tìm recursive ở các root hợp lý
    for root in roots:
        if not root.exists():
            continue
        for name in names:
            try:
                matches = list(root.rglob(name))
            except Exception:
                matches = []
            if matches:
                return matches[0]
    return None

train_path = auto_find_file(TRAIN_CSV, ["training_set.csv", "training_set(1).csv"])
val_path = auto_find_file(VAL_CSV, ["validation_set.csv"])
public_path = auto_find_file(PUBLIC_CSV, ["public_test.csv"])

# PRIVATE: ưu tiên file gốc. Chỉ fallback sang *_labeled để LẤY id,text,
# tuyệt đối bỏ label/noise_type nếu file đó tồn tại.
private_path = auto_find_file(
    PRIVATE_CSV,
    ["private_test.csv", "private_test_labeled.csv", "private_test_manual_reviewed_v2.csv"]
)

print("train  :", train_path)
print("val    :", val_path)
print("public :", public_path)
print("private:", private_path)

assert train_path is not None, "Không tìm thấy training_set.csv"
assert val_path is not None, "Không tìm thấy validation_set.csv"
assert public_path is not None, "Không tìm thấy public_test.csv"


## 3. Đọc dữ liệu và kiểm tra schema

In [ ]:

LABELS = ["CLEAN", "OFFENSIVE", "HATE"]
NOISE_LABELS = [
    "ORIGINAL",
    "NO_DIACRITICS",
    "TEENCODE",
    "CHAR_REPEAT",
    "PUNCT_NOISE",
    "OBFUSCATION",
    "MIXED",
]

label2id = {x:i for i,x in enumerate(LABELS)}
id2label = {i:x for x,i in label2id.items()}
noise2id = {x:i for i,x in enumerate(NOISE_LABELS)}
id2noise = {i:x for x,i in noise2id.items()}

def load_labeled(path):
    df = pd.read_csv(path)
    req = {"id", "text", "label", "noise_type"}
    assert req.issubset(df.columns), f"{path}: thiếu cột {req - set(df.columns)}"
    df = df[["id", "text", "label", "noise_type"]].copy()
    df["id"] = df["id"].astype(str)
    df["text"] = df["text"].fillna("").astype(str)
    assert set(df["label"]).issubset(LABELS)
    assert set(df["noise_type"]).issubset(NOISE_LABELS)
    return df

def load_test(path):
    if path is None:
        return None
    df = pd.read_csv(path)
    assert {"id", "text"}.issubset(df.columns), f"{path}: cần ít nhất id,text"
    # Chỉ lấy 2 cột này => không thể vô tình dùng nhãn có sẵn.
    df = df[["id", "text"]].copy()
    df["id"] = df["id"].astype(str)
    df["text"] = df["text"].fillna("").astype(str)
    return df

train_df = load_labeled(train_path)
val_df = load_labeled(val_path)
public_df = load_test(public_path)
private_df = load_test(private_path)

if USE_VALIDATION_FOR_FINAL_TRAIN:
    labeled_df = pd.concat([train_df, val_df], ignore_index=True)
else:
    labeled_df = train_df.copy()

print("train:", train_df.shape)
print("val:", val_df.shape)
print("labeled used:", labeled_df.shape)
print("public:", public_df.shape)
print("private:", None if private_df is None else private_df.shape)

display(labeled_df["label"].value_counts().to_frame("count"))
display(labeled_df["noise_type"].value_counts().to_frame("count"))



## 4. Group key để giảm leakage giữa augmentation/duplicate

Ta **không dùng `id`**. Group được tạo hoàn toàn từ `text`:
- lowercase;
- Unicode normalize;
- bỏ URL;
- bỏ dấu tiếng Việt;
- collapse character repeat;
- bỏ punctuation/space.

Mục tiêu là để những biến thể gần nhau có cơ hội ở cùng fold.


In [ ]:

URL_RE = re.compile(r"https?://\S+|www\.\S+", re.I)
REPEAT_RE = re.compile(r"(.)\1{2,}", re.UNICODE)

def strip_accents(s):
    s = unicodedata.normalize("NFD", s)
    return "".join(ch for ch in s if unicodedata.category(ch) != "Mn")

def make_group_key(text):
    s = unicodedata.normalize("NFKC", str(text)).lower()
    s = URL_RE.sub(" <url> ", s)
    s = strip_accents(s)
    s = REPEAT_RE.sub(r"\1\1", s)
    # Giữ chữ/số, bỏ ký hiệu và khoảng trắng để nối các bản obfuscation
    s = "".join(ch for ch in s if ch.isalnum())
    if not s:
        # fallback cho text toàn emoji/punctuation
        s = unicodedata.normalize("NFKC", str(text)).lower().strip()
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()

labeled_df["group"] = labeled_df["text"].map(make_group_key)
print("rows:", len(labeled_df), "| groups:", labeled_df["group"].nunique())
print("duplicate-group rows:", labeled_df.duplicated("group", keep=False).sum())


## 5. Feature bề mặt cho meta-model

In [ ]:

VI_DIACRITICS = set(
    "ăâđêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệ"
    "íìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ"
    "ĂÂĐÊÔƠƯÁÀẢÃẠẮẰẲẴẶẤẦẨẪẬÉÈẺẼẸẾỀỂỄỆ"
    "ÍÌỈĨỊÓÒỎÕỌỐỒỔỖỘỚỜỞỠỢÚÙỦŨỤỨỪỬỮỰÝỲỶỸỴ"
)

def meta_surface_features(texts):
    rows = []
    for text in texts:
        s = str(text)
        n = max(len(s), 1)
        alpha = max(sum(ch.isalpha() for ch in s), 1)
        punct = sum((not ch.isalnum()) and (not ch.isspace()) for ch in s)
        diac = sum(ch in VI_DIACRITICS for ch in s)
        repeats = len(re.findall(r"(.)\1{2,}", s.lower()))
        obfus = len(re.findall(r"(?<=[A-Za-zÀ-ỹ])[.*_~](?=[A-Za-zÀ-ỹ])", s))
        rows.append([
            math.log1p(len(s)),
            punct / n,
            diac / alpha,
            min(repeats, 10) / 10.0,
            min(obfus, 10) / 10.0,
        ])
    return np.asarray(rows, dtype=np.float32)

surface = meta_surface_features(labeled_df["text"])
print(surface.shape)


## 6. TF-IDF char + word và LinearSVM OOF

In [ ]:

class DualTfidf:
    def __init__(self):
        self.char = TfidfVectorizer(
            analyzer="char",
            ngram_range=CHAR_NGRAM,
            min_df=MIN_DF,
            max_features=CHAR_MAX_FEATURES,
            sublinear_tf=True,
            dtype=np.float32,
        )
        self.word = TfidfVectorizer(
            analyzer="word",
            ngram_range=WORD_NGRAM,
            min_df=MIN_DF,
            max_features=WORD_MAX_FEATURES,
            sublinear_tf=True,
            dtype=np.float32,
        )

    def fit(self, texts):
        texts = pd.Series(texts).fillna("").astype(str).tolist()
        self.char.fit(texts)
        self.word.fit(texts)
        return self

    def transform(self, texts):
        texts = pd.Series(texts).fillna("").astype(str).tolist()
        Xc = self.char.transform(texts)
        Xw = self.word.transform(texts)
        return sparse.hstack([Xc, Xw], format="csr", dtype=np.float32)

    def fit_transform(self, texts):
        self.fit(texts)
        return self.transform(texts)

y_hate = labeled_df["label"].map(label2id).to_numpy()
y_noise = labeled_df["noise_type"].map(noise2id).to_numpy()
groups = labeled_df["group"].to_numpy()

splitter = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)
folds = list(splitter.split(labeled_df, y_hate, groups))

svm_oof = np.zeros((len(labeled_df), len(LABELS)), dtype=np.float32)
noise_oof_scores = np.zeros((len(labeled_df), len(NOISE_LABELS)), dtype=np.float32)

svm_oof_path = WORK_DIR / "svm_oof.npy"
noise_oof_path = WORK_DIR / "noise_oof_scores.npy"

if REUSE_CACHE and svm_oof_path.exists() and noise_oof_path.exists():
    svm_oof = np.load(svm_oof_path)
    noise_oof_scores = np.load(noise_oof_path)
    print("Loaded cached SVM OOF.")
else:
    for fold, (tr_idx, va_idx) in enumerate(folds):
        print(f"\n===== TF-IDF fold {fold+1}/{N_FOLDS} =====")
        vec = DualTfidf()
        Xtr = vec.fit_transform(labeled_df.iloc[tr_idx]["text"])
        Xva = vec.transform(labeled_df.iloc[va_idx]["text"])

        hate_svm = LinearSVC(
            C=SVM_C_HATE,
            class_weight="balanced",
            random_state=SEED,
        )
        hate_svm.fit(Xtr, y_hate[tr_idx])
        svm_oof[va_idx] = hate_svm.decision_function(Xva).astype(np.float32)

        noise_svm = LinearSVC(
            C=SVM_C_NOISE,
            class_weight="balanced",
            random_state=SEED,
        )
        noise_svm.fit(Xtr, y_noise[tr_idx])
        noise_oof_scores[va_idx] = noise_svm.decision_function(Xva).astype(np.float32)

        del vec, Xtr, Xva, hate_svm, noise_svm
        gc.collect()

    np.save(svm_oof_path, svm_oof)
    np.save(noise_oof_path, noise_oof_scores)

print("SVM OOF hate Macro-F1:",
      f1_score(y_hate, svm_oof.argmax(1), average="macro"))
print("SVM OOF noise Macro-F1:",
      f1_score(y_noise, noise_oof_scores.argmax(1), average="macro"))


## 7. ViSoBERT weighted Trainer

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextClsDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = pd.Series(texts).fillna("").astype(str).tolist()
        self.labels = None if labels is None else np.asarray(labels, dtype=np.int64)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        item = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LENGTH,
        )
        if self.labels is not None:
            item["labels"] = int(self.labels[idx])
        return item

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def get_class_weights(y, n_classes):
    counts = np.bincount(y, minlength=n_classes).astype(np.float64)
    # sqrt-balanced: nhẹ hơn inverse-frequency đầy đủ
    weights = np.sqrt(len(y) / (n_classes * np.maximum(counts, 1)))
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, label_smoothing=0.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self._label_smoothing = float(label_smoothing)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = None
        if self.class_weights is not None:
            weight = self.class_weights.to(logits.device)
        loss = F.cross_entropy(
            logits,
            labels,
            weight=weight,
            label_smoothing=self._label_smoothing,
        )
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=1)
    return {"macro_f1": f1_score(labels, pred, average="macro")}

def make_training_args(output_dir, do_eval=True, epochs=EPOCHS):
    # Tương thích cả Transformers dùng eval_strategy và evaluation_strategy
    sig = inspect.signature(TrainingArguments.__init__).parameters
    kwargs = dict(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        logging_steps=100,
        seed=SEED,
        data_seed=SEED,
        fp16=torch.cuda.is_available(),
        report_to="none",
        save_total_limit=1,
        dataloader_num_workers=2,
    )
    if do_eval:
        if "eval_strategy" in sig:
            kwargs["eval_strategy"] = "epoch"
        else:
            kwargs["evaluation_strategy"] = "epoch"
        kwargs["save_strategy"] = "epoch"
        kwargs["load_best_model_at_end"] = True
        kwargs["metric_for_best_model"] = "macro_f1"
        kwargs["greater_is_better"] = True
    else:
        if "eval_strategy" in sig:
            kwargs["eval_strategy"] = "no"
        elif "evaluation_strategy" in sig:
            kwargs["evaluation_strategy"] = "no"
        kwargs["save_strategy"] = "no"
    return TrainingArguments(**kwargs)

def predict_proba_with_trainer(trainer, texts):
    ds = TextClsDataset(texts)
    logits = trainer.predict(ds).predictions
    return softmax(logits, axis=1).astype(np.float32)


## 8. ViSoBERT OOF

In [ ]:

viso_oof_path = WORK_DIR / "visobert_oof.npy"

if REUSE_CACHE and viso_oof_path.exists():
    viso_oof = np.load(viso_oof_path)
    print("Loaded cached ViSoBERT OOF.")
else:
    viso_oof = np.zeros((len(labeled_df), len(LABELS)), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(folds):
        fold_pred_path = WORK_DIR / f"visobert_fold{fold}_pred.npy"

        if REUSE_CACHE and fold_pred_path.exists():
            print(f"Fold {fold}: load cached prediction")
            viso_oof[va_idx] = np.load(fold_pred_path)
            continue

        print(f"\n===== ViSoBERT fold {fold+1}/{N_FOLDS} =====")
        model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,
            num_labels=len(LABELS),
            id2label=id2label,
            label2id=label2id,
        )

        tr_ds = TextClsDataset(labeled_df.iloc[tr_idx]["text"], y_hate[tr_idx])
        va_ds = TextClsDataset(labeled_df.iloc[va_idx]["text"], y_hate[va_idx])

        cw = get_class_weights(y_hate[tr_idx], len(LABELS)) if USE_CLASS_WEIGHTS else None

        trainer = WeightedTrainer(
            model=model,
            args=make_training_args(WORK_DIR / f"visobert_fold_{fold}", do_eval=True),
            train_dataset=tr_ds,
            eval_dataset=va_ds,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            class_weights=cw,
            label_smoothing=LABEL_SMOOTHING,
        )
        trainer.train()

        p = predict_proba_with_trainer(trainer, labeled_df.iloc[va_idx]["text"])
        viso_oof[va_idx] = p
        np.save(fold_pred_path, p)

        del trainer, model, tr_ds, va_ds
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    np.save(viso_oof_path, viso_oof)

print("ViSoBERT OOF Macro-F1:",
      f1_score(y_hate, viso_oof.argmax(1), average="macro"))


## 9. Train meta-model stacking

In [ ]:

# Chuyển noise decision scores -> softmax-like scores.
# Không coi đây là calibrated probability; chỉ là feature cho meta-model.
noise_oof_soft = softmax(noise_oof_scores, axis=1).astype(np.float32)

surface_oof = meta_surface_features(labeled_df["text"])

meta_X = np.hstack([
    viso_oof,          # 3
    svm_oof,           # 3
    noise_oof_soft,    # 7
    surface_oof,       # 5
]).astype(np.float32)

meta_model = Pipeline([
    ("scale", StandardScaler()),
    ("lr", LogisticRegression(
        C=META_C,
        max_iter=3000,
        class_weight="balanced",
        random_state=SEED,
    ))
])

meta_model.fit(meta_X, y_hate)
stack_oof_pred = meta_model.predict(meta_X)

hate_oof_f1 = f1_score(y_hate, stack_oof_pred, average="macro")
noise_oof_f1 = f1_score(y_noise, noise_oof_scores.argmax(1), average="macro")
competition_like_oof = 0.85 * hate_oof_f1 + 0.15 * noise_oof_f1

print("Stacking hate OOF Macro-F1 :", hate_oof_f1)
print("Noise OOF Macro-F1         :", noise_oof_f1)
print("0.85*hate + 0.15*noise      :", competition_like_oof)
print()
print(classification_report(y_hate, stack_oof_pred, target_names=LABELS, digits=4))

joblib.dump(meta_model, WORK_DIR / "meta_model.joblib")



> OOF score ở trên là để so sánh model. Vì group được normalize từ text, nó nghiêm ngặt hơn random row split, nhưng vẫn không thể đảm bảo khôi phục hoàn hảo source-family gốc.


## 10. Fit TF-IDF + SVM trên toàn bộ labeled data

In [ ]:

full_vec_path = WORK_DIR / "full_vectorizer.joblib"
full_hate_svm_path = WORK_DIR / "full_hate_svm.joblib"
full_noise_svm_path = WORK_DIR / "full_noise_svm.joblib"

if (
    REUSE_CACHE
    and full_vec_path.exists()
    and full_hate_svm_path.exists()
    and full_noise_svm_path.exists()
):
    full_vec = joblib.load(full_vec_path)
    full_hate_svm = joblib.load(full_hate_svm_path)
    full_noise_svm = joblib.load(full_noise_svm_path)
    print("Loaded cached full TF-IDF/SVM.")
else:
    full_vec = DualTfidf()
    X_all = full_vec.fit_transform(labeled_df["text"])

    full_hate_svm = LinearSVC(
        C=SVM_C_HATE,
        class_weight="balanced",
        random_state=SEED,
    )
    full_hate_svm.fit(X_all, y_hate)

    full_noise_svm = LinearSVC(
        C=SVM_C_NOISE,
        class_weight="balanced",
        random_state=SEED,
    )
    full_noise_svm.fit(X_all, y_noise)

    joblib.dump(full_vec, full_vec_path)
    joblib.dump(full_hate_svm, full_hate_svm_path)
    joblib.dump(full_noise_svm, full_noise_svm_path)

    del X_all
    gc.collect()

print("Full TF-IDF/SVM ready.")


## 11. Fit ViSoBERT trên toàn bộ labeled data

In [ ]:

full_viso_dir = WORK_DIR / "visobert_full"

if REUSE_CACHE and (full_viso_dir / "config.json").exists():
    print("Loading cached full ViSoBERT:", full_viso_dir)
    full_tokenizer = AutoTokenizer.from_pretrained(full_viso_dir)
    full_model = AutoModelForSequenceClassification.from_pretrained(full_viso_dir)
else:
    full_tokenizer = tokenizer
    full_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABELS),
        id2label=id2label,
        label2id=label2id,
    )

    full_ds = TextClsDataset(labeled_df["text"], y_hate)
    cw_full = get_class_weights(y_hate, len(LABELS)) if USE_CLASS_WEIGHTS else None

    full_trainer = WeightedTrainer(
        model=full_model,
        args=make_training_args(WORK_DIR / "visobert_full_train", do_eval=False),
        train_dataset=full_ds,
        data_collator=data_collator,
        class_weights=cw_full,
        label_smoothing=LABEL_SMOOTHING,
    )
    full_trainer.train()

    full_viso_dir.mkdir(parents=True, exist_ok=True)
    full_model.save_pretrained(full_viso_dir)
    tokenizer.save_pretrained(full_viso_dir)

    del full_trainer, full_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Full ViSoBERT ready.")


## 12. Inference public/private + stacking

In [ ]:

# Trainer inference riêng để dùng model full
infer_args = make_training_args(WORK_DIR / "infer_tmp", do_eval=False, epochs=1)
infer_trainer = WeightedTrainer(
    model=full_model,
    args=infer_args,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    class_weights=None,
    label_smoothing=0.0,
)

def infer_split(df, split_name):
    assert df is not None and {"id","text"}.issubset(df.columns)

    print(f"\nInference {split_name}: {len(df)} rows")

    # 1) TF-IDF
    X = full_vec.transform(df["text"])
    hate_svm_score = full_hate_svm.decision_function(X).astype(np.float32)
    noise_score = full_noise_svm.decision_function(X).astype(np.float32)
    noise_soft = softmax(noise_score, axis=1).astype(np.float32)

    # 2) ViSoBERT
    viso_prob = predict_proba_with_trainer(infer_trainer, df["text"])

    # 3) Surface features
    surf = meta_surface_features(df["text"])

    # 4) Stacking
    meta_test = np.hstack([
        viso_prob,
        hate_svm_score,
        noise_soft,
        surf,
    ]).astype(np.float32)

    hate_pred = meta_model.predict(meta_test)
    noise_pred = noise_score.argmax(1)

    out = pd.DataFrame({
        "id": df["id"].astype(str).values,
        "pred_label": [id2label[int(i)] for i in hate_pred],
        "pred_noise_type": [id2noise[int(i)] for i in noise_pred],
    })

    # Guard rails
    assert len(out) == len(df)
    assert out["id"].is_unique
    assert set(out["pred_label"]).issubset(LABELS)
    assert set(out["pred_noise_type"]).issubset(NOISE_LABELS)

    del X, hate_svm_score, noise_score, noise_soft, viso_prob, surf, meta_test
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out

public_out = infer_split(public_df, "public")
public_out.head()


In [ ]:

private_out = None
if private_df is not None:
    private_out = infer_split(private_df, "private")
    display(private_out.head())
else:
    print(
        "Chưa có private test. Khi private_test.csv được mở, "
        "đặt file vào DATA_DIR hoặc gán PRIVATE_CSV rồi chạy lại từ cell đọc private + inference. "
        "Không cần retrain nếu WORK_DIR/cache còn nguyên."
    )


## 13. Lưu CSV + ZIP đúng format

In [ ]:

import zipfile

OUT_DIR = WORK_DIR / "submissions"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_submission(df, csv_name, zip_name):
    csv_path = OUT_DIR / csv_name
    zip_path = OUT_DIR / zip_name

    df.to_csv(csv_path, index=False, encoding="utf-8")

    check = pd.read_csv(csv_path, dtype={"id": str})
    assert list(check.columns) == ["id", "pred_label", "pred_noise_type"]
    assert len(check) == len(df)
    assert check["id"].is_unique
    assert set(check["pred_label"]).issubset(LABELS)
    assert set(check["pred_noise_type"]).issubset(NOISE_LABELS)

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(csv_path, arcname=csv_path.name)

    print("Saved:", csv_path)
    print("Saved:", zip_path)
    return csv_path, zip_path

public_csv_path, public_zip_path = save_submission(
    public_out,
    "task1_public_output.csv",
    "task1_public_output.zip",
)

if private_out is not None:
    private_csv_path, private_zip_path = save_submission(
        private_out,
        "task1_private_output.csv",
        "task1_private_output.zip",
    )


In [ ]:

# Biến SVM decision score thành softmax-like distribution chỉ để weighted blend
svm_oof_soft = softmax(svm_oof, axis=1)

rows = []
for w_viso in np.arange(0.0, 1.01, 0.05):
    blend = w_viso * viso_oof + (1.0 - w_viso) * svm_oof_soft
    pred = blend.argmax(1)
    f1 = f1_score(y_hate, pred, average="macro")
    rows.append((w_viso, 1.0-w_viso, f1))

blend_df = pd.DataFrame(rows, columns=["w_visobert","w_svm","macro_f1"])
display(blend_df.sort_values("macro_f1", ascending=False).head(10))
print("Stacking OOF:", hate_oof_f1)
